In [ ]:
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
using Test
using AStarSearch

# Directions are seen as cartesian indexes so that they can be added to a position to get the next position
const UP = CartesianIndex(-1, 0)
const DOWN = CartesianIndex(1, 0)
const LEFT = CartesianIndex(0, -1)
const RIGHT = CartesianIndex(0, 1)
const DIRECTIONS = [UP, DOWN, LEFT, RIGHT]

# manhattan distance between positions in the maze matrix
manhattan(a::CartesianIndex, b::CartesianIndex) = sum(abs.((b - a).I))
# check to be in the maze and filter out moves that go into walls
function mazeneighbours(maze, p)
  res = CartesianIndex[]
  for d in DIRECTIONS
      n = p + d
      if 1 ≤ n[1] ≤ size(maze)[1] && 1 ≤ n[2] ≤ size(maze)[2] && !maze[n]
          push!(res, n)
      end
  end
  return res
end

function solvemaze(maze, start, goal)
  currentmazeneighbours(state) = mazeneighbours(maze, state)
  # Here you can use any of the exported search functions, they all share the same interface, but they won't use the heuristic and the cost
  return astar(currentmazeneighbours, start, goal, heuristic=manhattan)
end

# 0 = free cell, 1 = wall
maze = [0 0 1 0 0;
        0 1 0 0 0;
        0 1 0 0 1;
        0 0 0 1 1;
        1 0 1 0 0] .== 1
start = CartesianIndex(1, 1)
goal = CartesianIndex(1, 5)

res = solvemaze(maze, start, goal)
@test res.status == :success
@test res.path ==  CartesianIndex{2}[
    CartesianIndex(1, 1),
    CartesianIndex(2, 1),
    CartesianIndex(3, 1),
    CartesianIndex(4, 1),
    CartesianIndex(4, 2),
    CartesianIndex(4, 3),
    CartesianIndex(3, 3),
    CartesianIndex(2, 3),
    CartesianIndex(2, 4),
    CartesianIndex(1, 4),
    CartesianIndex(1, 5)]
@test res.cost == 10

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2
#### Caracterization of the circuit #####
ω_a = 25.338776456203686
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_a^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2pi /2

#Environment
κ_b=5/10.4; # 1/ns from Leon's paper

ω_d=ω_b # Drive frequency
ω_p=2*ω_a-ω_d # Pump Frequency

ε_p = 0.1
g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
ε_d = 10.12*g2#0.75*g2_max

β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
ε_d_p = ε_d*φ_b/ω_b
κ_b_p = κ_b / ω_b

ℵ = β_a_p*β_b_p/(4*κ_b_p)
ℶ = β_a_p*ε_d_p/(2*κ_b_p)

In [ ]:
# E_J = 37.12*2pi

# #Environment
# κ_b=1/10.4; # 1/ns from Leon's paper

# ω_d=ω_b # Drive frequency
# ω_p=2*ω_a-ω_d # Pump Frequency

# ε_p = 0.2
# g2 = besselj(1,ε_p)*E_J*φ_a^2*φ_b # (Leon's paper)
# ε_d = 10.12*g2#0.75*g2_max

# β_a_p, β_b_p = 2*E_J*sin(ε_p)*φ_a^2/(ω_a), 2*E_J*sin(ε_p)*φ_b^2/(ω_b)
# ε_d_p = ε_d*φ_b/ω_b
# κ_b_p = κ_b / ω_b

# ℵ = β_a_p*β_b_p/(4*κ_b_p)
# ℶ = β_a_p*ε_d_p/(2*κ_b_p)

In [ ]:
function H_vals(p_a, p_b, P_p_a, P_p_b, params)
    ϵ,β_a_p,β_b_p,ε_d_p,κ_b_p = params
    return @.(-β_a_p*(p_a*p_b)/sqrt(2))*P_p_a +
            2(-β_b_p*(-p_a^2)/2/sqrt(2)-ε_d_p/sqrt(2)-κ_b_p*p_b)*P_p_b +
            ϵ*(P_p_b^2)
end

function P_p_a_func(p_a, p_b, P_p_b, params, tol_den, tol_num)
    ϵ, β_a_p, β_b_p, ε_d_p, κ_b_p = params

    num = 2*(β_b_p*p_a^2/(2*sqrt(2)) - ε_d_p/sqrt(2) - κ_b_p*p_b)*P_p_b + ϵ*(P_p_b^2)
    den = β_a_p*(p_a*p_b)/sqrt(2)

    if abs(den) > tol_den
        return num/den
    elseif abs(den) < tol_den && abs(num) < tol_num
        return 0.0
    else
        return NaN
    end
end

In [ ]:



ϵ = 1e-3
params = (ϵ, β_a_p, β_b_p, ε_d_p, κ_b_p)

# fixed points
p_a_init, p_b_init = -sqrt(2*ε_d_p/β_b_p), 0.0
p_a_end, p_b_end = 0.0, -sqrt(0.5)*ε_d_p/κ_b_p

tol_den, tol_num = 1e-5, 1e-8

N_points = 100
N_points_P = 100

p_a_vals = range(-0.5, 0.1; length=N_points)
p_a_vals = sort!(unique(vcat(p_a_vals, p_a_init, p_a_end)))
p_b_vals = range(-0.1, 0.1; length=N_points)
p_b_vals = sort!(unique(vcat(p_b_vals, p_b_init, p_b_end)))
P_p_b_vals = range(-.2, .2, length=N_points_P)

starting_node = (
    searchsortedfirst(p_a_vals, p_a_init),
    searchsortedfirst(p_b_vals, p_b_init),
    div(N_points_P-1,2)+1
)

goal_node = (
    searchsortedfirst(p_a_vals, p_a_end),
    searchsortedfirst(p_b_vals, p_b_end),
    div(N_points_P-1,2)+1
)

invalid_nodes = Tuple{Int,Int,Int}[]
endpoint_nodes = Tuple{Int,Int,Int}[]
interior_nodes = Tuple{Int,Int,Int}[]

for i in 1:N_points, j in 1:N_points, k in 1:N_points_P

    pa = p_a_vals[i]
    pb = p_b_vals[j]
    Ppb = P_p_b_vals[k]

    res = P_p_a_func(pa, pb, Ppb, params, tol_den, tol_num)

    if isnan(res)

        push!(invalid_nodes, (i,j,k))

    elseif abs(pa*pb) < tol_den && abs(Ppb) < 0.1

        push!(endpoint_nodes, (i,j,k))

    else

        push!(interior_nodes, (i,j,k))

    end
end
println("Interior nodes: ", length(interior_nodes))
println("Endpoint nodes: ", length(endpoint_nodes))
println("Invalid nodes: ", length(invalid_nodes))

invalid_set = Set(invalid_nodes)
endpoint_set = Set(endpoint_nodes)


In [ ]:
function get_neighbors(node, invalid_set, endpoint_set,
                       N_points, N_points_P)

    i, j, k = node
    neighbors = Tuple{Int,Int,Int}[]

    # configuration moves
    config_moves = [
        (-1,0),(1,0),
        (0,-1),(0,1),
        (-1,-1),(-1,1),(1,-1),(1,1)
    ]

    momentum_moves = [-1,0,1]

    for (di,dj) in config_moves
        #if di < 0
        #    continue
        #end

        #if dj > 0
        #    continue
        #end

        for dk in momentum_moves

            ni = i + di
            nj = j + dj
            nk = k + dk

            if 1 ≤ ni ≤ N_points && 1 ≤ nj ≤ N_points && 1 ≤ nk ≤ N_points_P

                new_node = (ni,nj,nk)

                if new_node in invalid_set
                    continue
                end

                # restrict momentum variation near endpoint layer
                if node in endpoint_set && dk != 0
                    continue
                end

                push!(neighbors, new_node)

            end
        end
    end

    return neighbors
end

function edge_cost(node1, node2,
                   p_a_vals, p_b_vals, P_p_b_vals,
                   params, tol_den, tol_num)

    i1,j1,k1 = node1
    i2,j2,k2 = node2

    pa1 = p_a_vals[i1]
    pb1 = p_b_vals[j1]
    Ppb1 = P_p_b_vals[k1]

    pa2 = p_a_vals[i2]
    pb2 = p_b_vals[j2]
    Ppb2 = P_p_b_vals[k2]

    Ppa1 = P_p_a_func(pa1,pb1,Ppb1,params,tol_den,tol_num)
    Ppa2 = P_p_a_func(pa2,pb2,Ppb2,params,tol_den,tol_num)
    #Safety check
    if isnan(Ppa1) || isnan(Ppa2)
        return Inf
    end

    Δpa = pa2 - pa1
    Δpb = pb2 - pb1

    Ppa_mid = (Ppa1 + Ppa2)/2
    Ppb_mid = (Ppb1 + Ppb2)/2

    return Ppa_mid*Δpa + Ppb_mid*Δpb + 1e-1
end

In [ ]:
get_neighbors((0,13,35), invalid_set, endpoint_set, N_points, N_points_P)

In [ ]:
using DataStructures

function is_goal(node, goal_node, tol_index)

    i,j,_ = node
    gi,gj,_ = goal_node

    return abs(i-gi) ≤ tol_index && abs(j-gj) ≤ tol_index
end
using DataStructures

function dijkstra(start_node,
                  p_a_vals, p_b_vals, P_p_b_vals,
                  params, tol_den, tol_num,
                  invalid_set, endpoint_set,
                  N_points, N_points_P,
                  p_a_end, p_b_end)

    open_set = PriorityQueue{Tuple{Int,Int,Int},Float64}()

    # insert start node
    open_set[start_node] = 0.0

    g_score = Dict{Tuple{Int,Int,Int},Float64}()
    g_score[start_node] = 0.0

    came_from = Dict{Tuple{Int,Int,Int},Tuple{Int,Int,Int}}()

    visited = Set{Tuple{Int,Int,Int}}()
    c=0
    while !isempty(open_set)
        c+=1
        if mod(c,2e5)==0
            println(length(visited))
        end
        # node with smallest cost
        current = dequeue!(open_set)

        if current in visited
            continue
        end

        push!(visited, current)

        # goal test
        if is_goal(current, goal_node, 2)
            println("Goal reached")
            return reconstruct_path(came_from, current)
        end

        neighbors = get_neighbors(current,
                                  invalid_set,
                                  endpoint_set,
                                  N_points,
                                  N_points_P)

        for neighbor in neighbors

            if neighbor in visited
                continue
            end

            cost = edge_cost(current, neighbor,
                             p_a_vals, p_b_vals, P_p_b_vals,
                             params, tol_den, tol_num)

            tentative_g = g_score[current] + cost

            if !haskey(g_score, neighbor) || tentative_g < g_score[neighbor]

                g_score[neighbor] = tentative_g
                came_from[neighbor] = current

                # insert or update priority
                open_set[neighbor] = tentative_g
            end
        end
    end

    println("No path found")
    return nothing
end

function reconstruct_path(came_from, current)

    path = [current]

    while haskey(came_from, current)

        current = came_from[current]
        push!(path, current)

    end

    reverse!(path)

    return path
end

In [ ]:
path = dijkstra(starting_node,
                p_a_vals, p_b_vals, P_p_b_vals,
                params, tol_den, tol_num,
                invalid_set, endpoint_set,
                N_points, N_points_P,
                p_a_end, p_b_end)

In [ ]:
p_a_path = [p_a_vals[i] for (i,_,_) in path]
p_b_path = [p_b_vals[j] for (_,j,_) in path]
Ppb_path = [P_p_b_vals[k] for (_,_,k) in path]
Ppa_path = [
    P_p_a_func(pa,pb,Ppb,params,tol_den,tol_num)
    for (pa,pb,Ppb) in zip(p_a_path,p_b_path,Ppb_path)
];

In [ ]:
plot(Ppb_path, label="P_p_b")
plot!(Ppa_path, label="P_p_a", alpha=0.5)
xlabel!("Step on the path")

In [ ]:
plot(
    p_a_path, p_b_path,
    xlabel = "p_a",
    ylabel = "p_b",
    lw = 3,
    label = "optimal path",
)
scatter!([p_a_init],[p_b_init], label="start")
scatter!([p_a_end],[p_b_end], label="end")

In [ ]:
scatter3d(
    p_a_path, p_b_path, Ppb_path,
    xlabel = "p_a",
    ylabel = "p_b",
    zlabel= "P_p_b",
    ms = 1,
    label = "optimal path",
    marker_z = 1:length(path),
    markerstrokewidth = 0,
    c = :viridis
)

#scatter!([p_a_init],[p_b_init], label="start")
#scatter!([p_a_end],[p_b_end], label="end")